# Real Data Collector — YouTube Shorts
**Author:** Gustavo Pugliezi · companion to `viral_predictor.ipynb`

This notebook collects **real** short videos from YouTube and saves them as JSON, ready for Step 3 of the main notebook.

> **Intern analogy.** Before the intern can study, someone has to bring them a real pile of videos. This notebook is that someone: it searches YouTube Shorts by topic, writes down each video's public information, and "watches" the first and last frames to measure a few things a person would notice.

## What it collects for each video
| Source | Fields |
|---|---|
| Public metadata | title, description, duration, upload date, **views**, **channel subscribers**, likes |
| Automatic captions | what is said and **when** (timestamps), used for the hook |
| Video frames (analysed, then deleted) | face in the first second, number of cuts, whether the ending loops back to the start |

## How to run
1. Run it **on your own computer** (YouTube blocks most cloud servers, including Colab sometimes).
2. **Run → Run All Cells.** It takes roughly 30–60 minutes for ~300 videos and saves progress as it goes. If it stops, run it again: it continues where it left off.
3. When it finishes, `posts.json` and `new_posts.json` are in this folder. Then open `viral_predictor.ipynb`.

If YouTube answers *"Sign in to confirm you're not a bot"*, set `COOKIES_FROM_BROWSER = "chrome"` (or `"firefox"`, `"edge"`) in the settings cell, so the collector uses your browser's YouTube session.

In [ ]:
%pip install -q yt-dlp 'opencv-python-headless<5' numpy

## Settings

In [ ]:
TARGET_VIDEOS = 320          # videos with a settled view count (for training and testing)
TARGET_NEW_VIDEOS = 15       # very recent videos, with no final answer yet (for Step 10)
MIN_AGE_DAYS = 7             # Shorts get most of their views in the first week: younger videos are not "finished"
MAX_AGE_DAYS = 365
MAX_DURATION_S = 90
RESULTS_PER_QUERY = 60
COOKIES_FROM_BROWSER = None  # e.g. "chrome" if YouTube asks you to sign in

TOPIC_QUERIES = {
    "humor":     ["humor #shorts", "comédia #shorts", "pegadinha #shorts"],
    "education": ["curiosidades #shorts", "você sabia #shorts", "ciência #shorts"],
    "beauty":    ["maquiagem #shorts", "skincare #shorts", "cabelo tutorial #shorts"],
    "gaming":    ["minecraft #shorts", "free fire #shorts", "roblox #shorts"],
    "cooking":   ["receita fácil #shorts", "receita rápida #shorts", "doce fácil #shorts"],
    "sports":    ["futebol #shorts", "treino academia #shorts", "skate #shorts"],
}

## Collector code

In [ ]:
import json, os, random, re, time, shutil, tempfile
from datetime import datetime, date
import numpy as np
import cv2
import yt_dlp

WORK_DIR = tempfile.mkdtemp(prefix="shorts_")
FACE_MODEL = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


class BotCheck(Exception):
    pass


def ydl(extra=None):
    options = {"quiet": True, "no_warnings": True, "noprogress": True, "ignoreerrors": False}
    if COOKIES_FROM_BROWSER:
        options["cookiesfrombrowser"] = (COOKIES_FROM_BROWSER,)
    options.update(extra or {})
    return yt_dlp.YoutubeDL(options)


def search(query, n):
    """Fast search: only ids, titles, durations. No video page is opened."""
    with ydl({"extract_flat": True}) as y:
        result = y.extract_info(f"ytsearch{n}:{query}", download=False)
    return [e for e in result.get("entries", []) if e and e.get("id")]


def read_transcript(info):
    """Turns YouTube's automatic captions (json3) into lines like '[00:01.2] text'."""
    subs = info.get("requested_subtitles") or {}
    for lang in ("pt-orig", "pt", "en-orig", "en"):
        path = (subs.get(lang) or {}).get("filepath")
        if path and os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                events = json.load(f).get("events", [])
            lines = []
            for e in events:
                text = "".join(s.get("utf8", "") for s in e.get("segs", []) or []).replace("\n", " ").strip()
                if text:
                    t = e.get("tStartMs", 0) / 1000
                    lines.append(f"[{int(t // 60):02d}:{t % 60:04.1f}] {text}")
            return "\n".join(lines), lang
    return "", None


def analyse_video(path):
    """Watches the video frame by frame: face in the first second, cuts, and loop."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    previous_hist, first_small, last_small = None, None, None
    cuts, faces_first_second, frame_index = 0, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        small = cv2.resize(frame, (96, 160))
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256])
        cv2.normalize(hist, hist)
        if previous_hist is not None and cv2.compareHist(previous_hist, hist, cv2.HISTCMP_CORREL) < 0.55:
            cuts += 1                                   # the picture changed abruptly: a cut
        previous_hist = hist
        if first_small is None:
            first_small = small
        last_small = small
        if frame_index < fps and frame_index % max(1, int(fps // 4)) == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            found = FACE_MODEL.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=6,
                                                minSize=(gray.shape[1] // 8, gray.shape[1] // 8))
            faces_first_second += len(found) > 0
        frame_index += 1
    cap.release()
    if first_small is None:
        raise ValueError("video could not be read")
    a = cv2.calcHist([cv2.cvtColor(first_small, cv2.COLOR_BGR2HSV)], [0, 1], None, [16, 16], [0, 180, 0, 256])
    b = cv2.calcHist([cv2.cvtColor(last_small, cv2.COLOR_BGR2HSV)], [0, 1], None, [16, 16], [0, 180, 0, 256])
    loop_similarity = float(cv2.compareHist(cv2.normalize(a, a), cv2.normalize(b, b), cv2.HISTCMP_CORREL))
    return {"face_first_second": int(faces_first_second >= 2), "cuts": cuts,
            "loop_similarity": round(loop_similarity, 3), "has_loop": int(loop_similarity >= 0.9),
            "frames_analysed": frame_index}


def collect_one(video_id, topic):
    """Downloads one Short's data, analyses the video, deletes the file and returns a post."""
    for f in os.listdir(WORK_DIR):
        os.remove(os.path.join(WORK_DIR, f))
    options = {
        "format": "bv*[height<=480][ext=mp4]/bv*[height<=480]/b[height<=480]/wv*/w",
        "outtmpl": os.path.join(WORK_DIR, "%(id)s.%(ext)s"),
        "writeautomaticsub": True, "writesubtitles": False,
        "subtitleslangs": ["pt-orig", "pt", "en-orig", "en"], "subtitlesformat": "json3",
    }
    try:
        with ydl(options) as y:
            info = y.extract_info(f"https://www.youtube.com/shorts/{video_id}", download=True)
    except yt_dlp.utils.DownloadError as error:
        if "not a bot" in str(error) or "Sign in" in str(error):
            raise BotCheck(str(error))
        raise
    video_file = next((os.path.join(WORK_DIR, f) for f in os.listdir(WORK_DIR)
                       if f.startswith(video_id) and f.endswith((".mp4", ".webm", ".mkv"))), None)
    if not video_file:
        raise ValueError("no video file downloaded")
    transcript, language = read_transcript(info)
    return {
        "id": info["id"], "url": f"https://www.youtube.com/shorts/{info['id']}", "platform": "youtube_shorts",
        "topic": topic, "title": info.get("title") or "", "description": (info.get("description") or "")[:500],
        "transcript": transcript, "transcript_language": language,
        "duration_s": info.get("duration"), "posted_date": info.get("upload_date"),
        "followers": info.get("channel_follower_count"), "views": info.get("view_count"),
        "likes": info.get("like_count"), "channel": info.get("channel"),
        "collected_at": date.today().isoformat(),
        "video_analysis": analyse_video(video_file),
    }


def age_in_days(post):
    return (date.today() - datetime.strptime(post["posted_date"], "%Y%m%d").date()).days


def load(path):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return []


def save(path, posts):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(posts, f, ensure_ascii=False, indent=1)

print("Collector ready. Temporary folder:", WORK_DIR)

## Run the collection
Progress is saved after every video. Videos are skipped when the subscriber count is hidden, there are no views, the video is longer than the limit, or the age does not fit.

In [ ]:
posts, new_posts = load("posts.json"), load("new_posts.json")
seen = {p["id"] for p in posts + new_posts}
skipped = {}
per_topic = TARGET_VIDEOS // len(TOPIC_QUERIES) + 1
started = time.time()

try:
    for topic, queries in TOPIC_QUERIES.items():
        for query in queries:
            if sum(p["topic"] == topic for p in posts) >= per_topic and len(new_posts) >= TARGET_NEW_VIDEOS:
                break
            for entry in search(query, RESULTS_PER_QUERY):
                if sum(p["topic"] == topic for p in posts) >= per_topic and len(new_posts) >= TARGET_NEW_VIDEOS:
                    break
                if entry["id"] in seen or (entry.get("duration") or 0) > MAX_DURATION_S:
                    continue
                seen.add(entry["id"])
                time.sleep(random.uniform(2, 5))          # be polite: one video every few seconds
                try:
                    post = collect_one(entry["id"], topic)
                except BotCheck:
                    raise
                except Exception as error:
                    skipped[type(error).__name__] = skipped.get(type(error).__name__, 0) + 1
                    continue
                if not post["followers"] or not post["views"] or not post["posted_date"] or not post["duration_s"]:
                    skipped["missing data"] = skipped.get("missing data", 0) + 1
                    continue
                if post["duration_s"] > MAX_DURATION_S:
                    skipped["too long"] = skipped.get("too long", 0) + 1
                    continue
                age = age_in_days(post)
                if age < 3 and len(new_posts) < TARGET_NEW_VIDEOS:
                    new_posts.append({k: v for k, v in post.items() if k not in ("views", "likes")})
                    save("new_posts.json", new_posts)
                elif MIN_AGE_DAYS <= age <= MAX_AGE_DAYS and sum(p["topic"] == topic for p in posts) < per_topic:
                    posts.append(post)
                    save("posts.json", posts)
                else:
                    skipped["age"] = skipped.get("age", 0) + 1
                    continue
                minutes = (time.time() - started) / 60
                print(f"\r{len(posts):3d} videos | {len(new_posts):2d} new | {minutes:5.1f} min | last: {topic:9s} {post['title'][:40]!r}", end="")
except BotCheck:
    print("\n\nYouTube asked to confirm you are not a bot. Set COOKIES_FROM_BROWSER (see the top of this notebook) "
          "and run again: the collection continues where it stopped.")
finally:
    shutil.rmtree(WORK_DIR, ignore_errors=True)

print(f"\n\nSaved {len(posts)} videos to posts.json and {len(new_posts)} recent videos to new_posts.json")
print("Skipped:", skipped)

## Quick check of what was collected

In [ ]:
import pandas as pd
posts = load("posts.json")
if not posts:
    raise SystemExit("No videos collected yet. Run the collection cell again (see the bot-check tip at the top).")
summary = pd.DataFrame([{"topic": p["topic"], "views": p["views"], "followers": p["followers"], "duration_s": p["duration_s"],
                         "has_transcript": bool(p["transcript"]), **p["video_analysis"]} for p in posts])
print(f"{len(summary)} videos")
display(summary.groupby("topic").size().rename("videos").to_frame().T)
summary.describe().round(2)